<a href="https://colab.research.google.com/github/saithrr/Telecom_X_-SRR/blob/main/TELECOM_X.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importe y Normalización de Datos

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import requests
import json
import datetime
import chardet
from scipy.stats import pointbiserialr

In [3]:
url = 'https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json'
respuesta= requests.get(url)
data= respuesta.json()
df= pd.json_normalize(data)
df.head(200)

,customerID,Churn,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,...,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.60,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.90,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.90,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.00,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.90,267.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0306-JAELE,Yes,Male,0,No,No,5,Yes,No,Fiber optic,...,No,No,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,96.10,453.4
196,0307-BCOPK,No,Female,0,No,No,16,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.05,326.65
197,0308-GIQJT,No,Male,1,No,No,71,Yes,Yes,Fiber optic,...,Yes,No,Yes,Yes,Yes,One year,No,Bank transfer (automatic),105.75,7382.85
198,0308-IVGOK,No,Female,0,No,No,11,No,No phone service,DSL,...,Yes,Yes,Yes,No,No,One year,Yes,Credit card (automatic),40.40,422.6


## Conteo de Datos Nulos

In [4]:
null_counts = df.isnull().sum()
null_counts_df = pd.DataFrame({'Columna': null_counts.index, 'Conteo_Nulos': null_counts.values})
display(null_counts_df[null_counts_df['Conteo_Nulos'] > 0])

if (null_counts == 0).all():
    print("No se encontraron datos nulos en ninguna columna.")
else:
    print("Se encontraron datos nulos en las columnas listadas anteriormente.")

,Columna,Conteo_Nulos


No se encontraron datos nulos en ninguna columna.


### Revisar Tipos de Datos del DataFrame

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7267 non-null   object 
 1   Churn                      7267 non-null   object 
 2   customer.gender            7267 non-null   object 
 3   customer.SeniorCitizen     7267 non-null   int64  
 4   customer.Partner           7267 non-null   object 
 5   customer.Dependents        7267 non-null   object 
 6   customer.tenure            7267 non-null   int64  
 7   phone.PhoneService         7267 non-null   object 
 8   phone.MultipleLines        7267 non-null   object 
 9   internet.InternetService   7267 non-null   object 
 10  internet.OnlineSecurity    7267 non-null   object 
 11  internet.OnlineBackup      7267 non-null   object 
 12  internet.DeviceProtection  7267 non-null   object 
 13  internet.TechSupport       7267 non-null   objec

## Revisión de Datos Duplicados


In [6]:
duplicate_rows = df[df.duplicated()]

if not duplicate_rows.empty:
    print(f"Se encontraron {len(duplicate_rows)} filas duplicadas:")
    display(duplicate_rows)
else:
    print("No se encontraron filas duplicadas en el DataFrame.")

No se encontraron filas duplicadas en el DataFrame.


In [7]:
df.shape

(7267, 21)

In [8]:
print('--- Verificación de valores en Blanco ---')

# Verificar NaN (valores nulos) en todo el DataFrame
nan_counts = df.isnull().sum()
nan_cols_with_data = nan_counts[nan_counts > 0]

if not nan_cols_with_data.empty:
    print('\nSe encontraron valores NaN en las siguientes columnas:')
    display(nan_cols_with_data.reset_index(name='Conteo_NaN').rename(columns={'index': 'Columna'}))
else:
    print('\nNo se encontraron valores NaN en ninguna columna.')

# Verificar cadenas vacías o espacios en blanco en columnas de tipo 'object'
empty_string_counts = pd.Series(dtype='int')
for col in df.select_dtypes(include='object').columns:
    count = df[df[col].astype(str).str.strip() == ''].shape[0]
    if count > 0:
        empty_string_counts[col] = count

if not empty_string_counts.empty:
    print('\nSe encontraron cadenas vacías o solo espacios en blanco en las siguientes columnas:')
    display(empty_string_counts.reset_index(name='Conteo_Vacios').rename(columns={'index': 'Columna'}))
else:
    print('\nNo se encontraron cadenas vacías o solo espacios en blanco en columnas de tipo object.')

print('\n--- Verificación completada ---')

--- Verificación de valores en Blanco ---

No se encontraron valores NaN en ninguna columna.

Se encontraron cadenas vacías o solo espacios en blanco en las siguientes columnas:


,Columna,Conteo_Vacios
0,Churn,224
1,account.Charges.Total,11



--- Verificación completada ---


# Transformando DataFrame

*Reemplazar datos de la columna 'account.Charges.Total' a tipo FLOAT y los vacios por valor cero.

In [9]:
df['account.Charges.Total'] = df['account.Charges.Total'].replace(' ', '0')
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'])
print("La columna 'account.Charges.Total' ha sido transformada a tipo numérico.")

df.info()

La columna 'account.Charges.Total' ha sido transformada a tipo numérico.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7267 non-null   object 
 1   Churn                      7267 non-null   object 
 2   customer.gender            7267 non-null   object 
 3   customer.SeniorCitizen     7267 non-null   int64  
 4   customer.Partner           7267 non-null   object 
 5   customer.Dependents        7267 non-null   object 
 6   customer.tenure            7267 non-null   int64  
 7   phone.PhoneService         7267 non-null   object 
 8   phone.MultipleLines        7267 non-null   object 
 9   internet.InternetService   7267 non-null   object 
 10  internet.OnlineSecurity    7267 non-null   object 
 11  internet.OnlineBackup      7267 non-null   object 
 12  internet.DeviceProtection  7267

In [10]:
for col in df.columns:
  print(f'{col}: {df[col].nunique()}')
  if df[col].nunique() < 100:
    print(df[col].unique())
    print('-' * 100)

customerID: 7267
Churn: 3
['No' 'Yes' '']
----------------------------------------------------------------------------------------------------
customer.gender: 2
['Female' 'Male']
----------------------------------------------------------------------------------------------------
customer.SeniorCitizen: 2
[0 1]
----------------------------------------------------------------------------------------------------
customer.Partner: 2
['Yes' 'No']
----------------------------------------------------------------------------------------------------
customer.Dependents: 2
['Yes' 'No']
----------------------------------------------------------------------------------------------------
customer.tenure: 73
[ 9  4 13  3 71 63  7 65 54 72  5 56 34  1 45 50 23 55 26 69 11 37 49 66
 67 20 43 59 12 27  2 25 29 14 35 64 39 40  6 30 70 57 58 16 32 33 10 21
 61 15 44 22 24 19 47 62 46 52  8 60 48 28 41 53 68 51 31 36 17 18 38 42
  0]
-----------------------------------------------------------------------

* Eliminacion de datos vacios de la columna 'Churn'.

In [11]:
df = df[df['Churn'] != '']
print("Datos vacíos eliminados de la columna 'Churn'.")
df['Churn'].unique()

Datos vacíos eliminados de la columna 'Churn'.


array(['No', 'Yes'], dtype=object)

In [11]:
* Traduciendo el nombre de las columnas

In [15]:
columnas = {'customerID': 'id',
    'customer.gender': 'genero',
    'customer.SeniorCitizen': 'tiene +60',
    'customer.Partner': 'posee_pareja',
    'customer.Dependents': 'posee_dependientes',
    'customer.tenure': 'tiempo_contrato',
    'phone.PhoneService': 'servicio_telefono',
    'phone.MultipleLines': 'multiples_lineas',
    'internet.InternetService': 'tipo_internet',
    'internet.OnlineSecurity': 'seguridad_online',
    'internet.OnlineBackup': 'backup_online',
    'internet.DeviceProtection': 'proteccion_dispositivo',
    'internet.TechSupport': 'soporte_tecnico',
    'internet.StreamingTV': 'streaming_tv',
    'internet.StreamingMovies': 'streaming_peliculas',
    'account.Contract': 'tipo_contrato',
    'account.PaperlessBilling': 'factura_digital',
    'account.PaymentMethod': 'metodo_pago',
    'account.Charges.Monthly': 'valor_mensal',
    'account.Charges.Total': 'total_cobrado'
}
df = df.rename(columns= columnas)


In [ ]:
* Traduciendo los valores de las columnas

In [24]:
df['Churn'] = df['Churn'].replace({'No': 'No', 'Yes': 'Sí'})
df['genero'] = df['genero'].replace({'Female': 'Femenino', 'Male': 'Masculino'})
df['posee_pareja'] = df['posee_pareja'].replace({'Yes': 'Sí', 'No': 'No'})
df['posee_dependientes'] = df['posee_dependientes'].replace({'Yes': 'Sí', 'No': 'No'})
df['servicio_telefono'] = df['servicio_telefono'].replace({'Yes': 'Sí', 'No': 'No'})
df['multiples_lineas'] = df['multiples_lineas'].replace({'No': 'No', 'Yes': 'Sí', 'No phone service': 'Sin servicio de teléfono'})
df['tipo_internet'] = df['tipo_internet'].replace({'No': 'No'})
df['tipo_contrato'] = df['tipo_contrato'].replace({'One year': 'Anual', 'Month-to-month': 'Mensual', 'Two year': 'Bienal'})
df['tipo_internet'] = df['tipo_internet'].replace({'Fiber optic': 'Fibra óptica'})

In [19]:
columnas_a_traducir = ['seguridad_online', 'backup_online', 'proteccion_dispositivo', 'soporte_tecnico', 'streaming_tv', 'streaming_peliculas']
mapeo = {'No': 'No', 'Yes': 'Sí', 'No internet service': 'Sin servicio de internet'}

for col in columnas_a_traducir:
    df[col] = df[col].replace(mapeo)


In [21]:
df['metodo_pago'] = df['metodo_pago'].replace({
    'Mailed check': 'Cheque enviado por correo',
    'Electronic check': 'Cheque electrónico',
    'Credit card (automatic)': 'Tarjeta de crédito (automático)',
    'Bank transfer (automatic)': 'Transferencia bancaria (automática)'
})

In [22]:
df

,id,Churn,genero,tiene +60,posee_pareja,posee_dependientes,tiempo_contrato,servicio_telefono,multiples_lineas,tipo_internet,...,backup_online,proteccion_dispositivo,soporte_tecnico,streaming_tv,streaming_peliculas,tipo_contrato,factura_digital,metodo_pago,valor_mensal,total_cobrado
0,0002-ORFBO,No,Femenino,0,Sí,Sí,9,Sí,No,DSL,...,Sí,No,Sí,Sí,No,Anual,Yes,Cheque enviado por correo,65.60,593.30
1,0003-MKNFE,No,Masculino,0,No,No,9,Sí,Sí,DSL,...,No,No,No,No,Sí,Mensual,No,Cheque enviado por correo,59.90,542.40
2,0004-TLHLJ,Sí,Masculino,0,No,No,4,Sí,No,Fiber optic,...,No,Sí,No,No,No,Mensual,Yes,Cheque electrónico,73.90,280.85
3,0011-IGKFF,Sí,Masculino,1,Sí,No,13,Sí,No,Fiber optic,...,Sí,Sí,No,Sí,Sí,Mensual,Yes,Cheque electrónico,98.00,1237.85
4,0013-EXCHZ,Sí,Femenino,1,Sí,No,3,Sí,No,Fiber optic,...,No,No,Sí,Sí,No,Mensual,Yes,Cheque enviado por correo,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7262,9987-LUTYD,No,Femenino,0,No,No,13,Sí,No,DSL,...,No,No,Sí,No,No,Anual,No,Cheque enviado por correo,55.15,742.90
7263,9992-RRAMN,Sí,Masculino,0,Sí,No,22,Sí,Sí,Fiber optic,...,No,No,No,No,Sí,Mensual,Yes,Cheque electrónico,85.10,1873.70
7264,9992-UJOEL,No,Masculino,0,No,No,2,Sí,No,DSL,...,Sí,No,No,No,No,Mensual,Yes,Cheque enviado por correo,50.30,92.75
7265,9993-LHIEB,No,Masculino,0,Sí,Sí,67,Sí,No,DSL,...,No,Sí,Sí,No,Sí,Bienal,No,Cheque enviado por correo,67.85,4627.65


In [25]:
for col in df.columns:
  print(f'{col}: {df[col].nunique()}')
  if df[col].nunique() < 100:
    print(df[col].unique())
    print('-' * 100)

id: 7043
Churn: 2
['No' 'Sí']
----------------------------------------------------------------------------------------------------
genero: 2
['Femenino' 'Masculino']
----------------------------------------------------------------------------------------------------
tiene +60: 2
[0 1]
----------------------------------------------------------------------------------------------------
posee_pareja: 2
['Sí' 'No']
----------------------------------------------------------------------------------------------------
posee_dependientes: 2
['Sí' 'No']
----------------------------------------------------------------------------------------------------
tiempo_contrato: 73
[ 9  4 13  3 71 63  7 65 54 72  5 56 34  1 45 50 23 55 26 69 37 49 66 67
 20 43 59 12 27  2 25 29 14 35 64 39 40 11  6 30 70 57 58 16 32 33 10 21
 61 15 44 22 24 19 47 62 46 52  8 60 48 28 41 53 68 31 36 17 18 51 38 42
  0]
----------------------------------------------------------------------------------------------------
serv